## Refined Frequency Branch alone , on combined dataset

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split , Dataset
import timm
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)


print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

PyTorch: 2.11.0+cu128
GPU: NVIDIA L4


##Imports and Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
import os
import shutil

with zipfile.ZipFile('/content/drive/MyDrive/data_combined_8k.zip', 'r') as z:
    for member in z.namelist():
        fixed_path = member.replace('\\', '/')
        target = os.path.join('/content', fixed_path)
        if member.endswith('\\') or '.' not in os.path.basename(fixed_path):
            os.makedirs(target, exist_ok=True)
        else:
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(member) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)

print("Real frames:", len(os.listdir('/content/data_combined_8k/real')))
print("Fake frames:", len(os.listdir('/content/data_combined_8k/fake')))

Mounted at /content/drive
Real frames: 8000
Fake frames: 8000


##Dataset preparation

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.RandomHorizontalFlip(),   #purposely disabled certain augmentation
    #transforms.RandomRotation(10),
    #transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

test_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        image = self.subset.dataset.loader(path)
        return self.transform(image), label


full_dataset = ImageFolder('/content/data_combined_8k', transform=None)
print("Classes:", full_dataset.classes)
print("Total:", len(full_dataset))

train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_sub, val_sub, test_sub = random_split(full_dataset, [train_size, val_size, test_size])

train_set = TransformSubset(train_sub, train_transform)
val_set = TransformSubset(val_sub, test_transform)
test_set = TransformSubset(test_sub, test_transform)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=2)

print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")

Classes: ['fake', 'real']
Total: 16000
Train: 11200, Val: 2400, Test: 2400


##Defining the refined branch

In [ ]:

class FrequencyBranch(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(6, 16, kernel_size=3, padding=1),  # 6 channels: RGB magnitude + RGB phase
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )

    def forward(self, x):

        # FFT now separately on R, G and B channels
        fft = torch.fft.fft2(x)
        fft = torch.fft.fftshift(fft, dim=(-2, -1))

        # Magnitude for each RGB channel
        magnitude = torch.abs(fft)
        magnitude = torch.log1p(magnitude)

        magnitude = (
            magnitude -
            magnitude.mean(dim=(-2, -1), keepdim=True)
        ) / (
            magnitude.std(dim=(-2, -1), keepdim=True) + 1e-6
        )

        # Phase for each RGB channel
        phase = torch.angle(fft)
        phase = phase / torch.pi

        # 3 magnitude + 3 phase = 6 input channels
        freq_input = torch.cat(
            [magnitude, phase],
            dim=1
        )

        out = self.features(freq_input)
        out = self.classifier(out)

        return out


class FrequencyOnlyDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.freq_branch = FrequencyBranch()
        self.classifier = nn.Linear(128, 2)

    def forward(self, x):
        features = self.freq_branch(x)
        return self.classifier(features)

model = FrequencyOnlyDetector().cuda()
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Trainable parameters: 155,698


##optimizer and scheduler

In [ ]:
num_epochs = 30

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

##Training and validation functions

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.cuda(), labels.cuda()
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss/len(loader), 100.*correct/total

def val_epoch(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.cuda(), labels.cuda()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    return running_loss/len(loader), 100.*correct/total

print("Training functions ready")

Training functions ready


##Training loop

extra 15 epochs to assess learning ability

In [ ]:
best_val_acc = 0
train_losses, val_losses, train_accs, val_accs = [], [], [], []

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = val_epoch(model, val_loader, criterion)
    scheduler.step()

    train_losses.append(train_loss); val_losses.append(val_loss)
    train_accs.append(train_acc); val_accs.append(val_acc)
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_model_freqonlytestnew30ep.pth')
        print("  -> Best model saved")

print(f"\nBest validation accuracy: {best_val_acc:.2f}%")

Epoch 1/30 | Train Loss: 0.6832 | Train Acc: 54.52% | Val Loss: 0.6836 | Val Acc: 54.12%
  -> Best model saved
Epoch 2/30 | Train Loss: 0.6824 | Train Acc: 55.16% | Val Loss: 0.6829 | Val Acc: 54.38%
  -> Best model saved
Epoch 3/30 | Train Loss: 0.6810 | Train Acc: 55.06% | Val Loss: 0.6847 | Val Acc: 52.96%
Epoch 4/30 | Train Loss: 0.6799 | Train Acc: 55.22% | Val Loss: 0.6838 | Val Acc: 54.29%
Epoch 5/30 | Train Loss: 0.6767 | Train Acc: 55.88% | Val Loss: 0.6785 | Val Acc: 55.96%
  -> Best model saved
Epoch 6/30 | Train Loss: 0.6773 | Train Acc: 55.50% | Val Loss: 0.6762 | Val Acc: 56.83%
  -> Best model saved
Epoch 7/30 | Train Loss: 0.6749 | Train Acc: 56.38% | Val Loss: 0.6750 | Val Acc: 56.38%
Epoch 8/30 | Train Loss: 0.6737 | Train Acc: 56.47% | Val Loss: 0.7145 | Val Acc: 53.46%
Epoch 9/30 | Train Loss: 0.6738 | Train Acc: 56.31% | Val Loss: 0.6915 | Val Acc: 54.42%
Epoch 10/30 | Train Loss: 0.6723 | Train Acc: 56.79% | Val Loss: 0.6729 | Val Acc: 56.29%
Epoch 11/30 | Train L